In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from classification_Enes import CLASSES, load_groups, find_scans, make_pairs, PairDataset, Classifier
from UNet_Base_Enes import UNet

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
FREEZE = True
EPOCHS = 30
BATCH_SIZE = 16
LR = 0.001
ENCODER_LR = 0.0001
FOLDS = 5
SEED = 0
PRETRAINED = '../semi-supervised/models/mean_teacher/last.pth'
if FREEZE:
    MODEL_PATH = 'models/classifier_frozen.pth'
else:
    MODEL_PATH = 'models/classifier_finetune.pth'

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
groups = load_groups('groupes.csv')
scans = find_scans([DATA_DIR + 'train/Img', DATA_DIR + 'train/Img-Unlabeled'])
patients = sorted(groups, key=lambda p: (groups[p], p))
fold = {p: i % FOLDS for i, p in enumerate(patients)}

print(len(patients), 'patients,', len(make_pairs(scans, patients)), 'paires ED/ES')
for c, name in enumerate(CLASSES):
    print(name, sum(groups[p] == c for p in patients))

In [ ]:
def new_model():
    unet = UNet(num_classes=4)
    unet.load_state_dict(torch.load(PRETRAINED, map_location='cpu'))
    model = Classifier(unet).to(device)
    if FREEZE:
        for p in model.encoder.parameters():
            p.requires_grad = False
    return model


def train(model, pairs):
    loader = DataLoader(PairDataset(pairs, groups, augment=True), batch_size=BATCH_SIZE, shuffle=True)
    params = [{'params': model.head.parameters(), 'lr': LR}]
    if not FREEZE:
        params.append({'params': model.encoder.parameters(), 'lr': ENCODER_LR})
    optimizer = torch.optim.Adam(params)
    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        model.train()
        if FREEZE:
            model.encoder.eval()
        losses = []
        for ed, es, labels, _ in loader:
            out = model(ed.to(device), es.to(device))
            loss = ce_loss(out, labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print('   Epoch {} | loss {:.4f}'.format(epoch, np.mean(losses)))
    return model


@torch.no_grad()
def predict_patients(model, pairs):
    model.eval()
    loader = DataLoader(PairDataset(pairs, groups), batch_size=BATCH_SIZE)
    probs = {}
    for ed, es, _, patient in loader:
        out = torch.softmax(model(ed.to(device), es.to(device)), dim=1).cpu()
        for k in range(len(patient)):
            probs.setdefault(patient[k].item(), []).append(out[k])
    return {p: torch.stack(v).mean(dim=0).argmax().item() for p, v in probs.items()}

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
predictions = {}

for k in range(FOLDS):
    train_patients = [p for p in patients if fold[p] != k]
    test_patients = [p for p in patients if fold[p] == k]
    print('Pli {} : {} patients en entraînement, {} en test'.format(k, len(train_patients), len(test_patients)))
    model = train(new_model(), make_pairs(scans, train_patients))
    pred = predict_patients(model, make_pairs(scans, test_patients))
    predictions.update(pred)
    good = sum(pred[p] == groups[p] for p in test_patients)
    print('Pli {} : {} / {} patients bien classés'.format(k, good, len(test_patients)))

In [ ]:
good = sum(predictions[p] == groups[p] for p in patients)
print('Précision (validation croisée) : {} / {} = {:.3f}'.format(good, len(patients), good / len(patients)))

matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for p in patients:
    matrix[groups[p], predictions[p]] += 1

plt.imshow(matrix, cmap='Blues')
for t in range(len(CLASSES)):
    for c in range(len(CLASSES)):
        plt.text(c, t, matrix[t, c], ha='center', va='center')
plt.xticks(range(len(CLASSES)), CLASSES)
plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel('Groupe prédit')
plt.ylabel('Vrai groupe')
plt.show()

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
model = train(new_model(), make_pairs(scans, patients))
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), MODEL_PATH)